# Machining Sounds

## Part C: Machining

*A FAB26 workshop lesson, companion to* Machining Dynamics: Jupyter Notebook Edition *(DOI: forthcoming)*

**Authors:** Tony L. Schmitz and Michael F. Gomez

> *"Chatter is the most obscure and delicate of all problems facing the machinist."*
> F. W. Taylor, *On the Art of Cutting Metals* (1907)


Part A built the vocabulary of vibrating systems and Part B built the machinery of sound analysis. Part C puts both on a milling machine. We describe the milling process following the treatment of Chap. 4 of Schmitz and Smith [1], predict what a stable cut must sound like using only Part B, and meet the mechanism that can break that prediction: regeneration, the feedback loop behind chatter. The stability lobe diagram is then developed twice, by two analytical solutions presented side by side, and tested against the Sect. 4.4 time domain simulation of [1], which we port in full with every parameter exposed. The Part closes with the practical loop the whole series has been building toward: listen to a cut, identify the chatter frequency, and choose a better spindle speed.

The purposes of Part C are to:

- Describe the milling operation: kinematics, chip thickness, cutting force, and the tooth passing frequency.
- Predict and synthesize the sound of a stable cut from its ingredients.
- Explain regeneration, the mechanism of self-excited vibration in milling.
- Compute stability lobe diagrams by two analytical solutions, the average tooth angle approach and the Fourier series (zero-order) approach, stated plainly with their assumptions.
- Port and exercise the p_4_10_1.m milling time domain simulation of [1] with a full parameter panel.
- Build a stability map directly from time domain simulations and compare it with the analytical boundaries.
- Detect chatter in the frequency content of a milling signal and select a better spindle speed from the chatter frequency.

Parts A and B are assumed. Equations taken from [1] keep their original numbers.


### Nomenclature

The following symbols are used consistently throughout the lesson, matching the notation of [1]:

| Symbol | Meaning | Units |
|--------|---------|-------|
| $\Omega$ | spindle speed | rpm |
| $N_t$ | number of teeth | dimensionless |
| $f_{tooth}$ | tooth passing frequency, $\Omega N_t / 60$ | Hz |
| $\tau$ | tooth period, $60 / (\Omega N_t)$ | s |
| $\phi$ | cutter rotation angle | deg |
| $\phi_s$, $\phi_e$ | cut start and exit angles | deg |
| $f_t$ | feed per tooth | mm/tooth |
| $h$ | instantaneous chip thickness | mm |
| $b$ | axial depth of cut (chip width) | mm |
| $a$ | radial depth of cut | mm |
| $r$, $d$ | tool radius and diameter | mm |
| $k_t$, $k_n$ | tangential and normal cutting force coefficients | N/mm² |
| $K_s$, $\beta$ | specific force, $\sqrt{k_t^2 + k_n^2}$, and force angle | N/mm², deg |
| $F_t$, $F_n$ | tangential and normal force components | N |
| $F_x$, $F_y$ | fixed-frame force components | N |
| $n$ | vibration along the surface normal | m |
| $f_n$, $k$, $\zeta$ | modal parameters per direction (Part A) | Hz, N/m, dimensionless |
| $\mu_x$, $\mu_y$ | directional orientation factors | dimensionless |
| $b_{lim}$ | limiting axial depth of cut | mm |
| $N$ | stability lobe number | dimensionless |
| $\varepsilon$ | tooth-to-tooth undulation phase | rad |
| $f_c$ | chatter frequency | Hz |

In code, $\Omega$ appears as `omega_rpm`, $\phi$ as `phi`, and the modal symbols keep their Part A names.


## Setup: Install and Import Libraries

Run the install cell once. If this is the first install on your machine, restart the kernel afterward (Kernel menu, Restart), then continue from the import cell.


In [ ]:
%pip install -q numpy matplotlib plotly


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Audio, display

AUDIO_RATE = 48000  # Hz, sample rate for all synthesized sound

print('Libraries loaded.')


The utility functions carried over from Parts A and B are defined in the collapsed cell below and documented in the Appendix. The analytical stability solutions and the time domain simulation are not utilities; they are the content of Sects. C.4 and C.5 and appear there in full view.


In [ ]:
# --- Utility functions used throughout the lesson (documented in the Appendix) ---
def fade(x, ms=15, fs=AUDIO_RATE):
    """Apply a short raised-cosine fade-in and fade-out so audio clips start
    and stop without clicks."""
    n = int(ms * 1e-3 * fs)
    ramp = 0.5 * (1 - np.cos(np.pi * np.arange(n) / n))
    y = x.copy()
    y[:n] *= ramp
    y[-n:] *= ramp[::-1]
    return y

def compute_spectrum(x, fs, fmax=4000):
    """Amplitude spectrum of signal x on a linear scale, normalized to a peak
    value of one. Returns the frequency vector (Hz, limited to fmax) and the
    normalized spectrum."""
    x = np.asarray(x, dtype=float)
    w = np.hanning(len(x))
    X = np.fft.rfft((x - x.mean()) * w)
    f = np.fft.rfftfreq(len(x), 1 / fs)
    amp = 2 * np.abs(X) / np.sum(w)
    amp = amp / amp.max()
    sel = f <= fmax
    return f[sel], amp[sel]

def compute_spectrogram(x, fs, nfft=2048, hop=512, fmax=4000):
    """Short-time amplitude spectrogram on a linear scale, normalized to a
    peak value of one. Returns (t_frames, f, S) with one column per frame."""
    w = np.hanning(nfft)
    n_frames = 1 + max(0, (len(x) - nfft) // hop)
    f = np.fft.rfftfreq(nfft, 1 / fs)
    sel = f <= fmax
    S = np.empty((int(sel.sum()), n_frames))
    for j in range(n_frames):
        S[:, j] = np.abs(np.fft.rfft(x[j * hop:j * hop + nfft] * w))[sel]
    S /= S.max()
    t_frames = (np.arange(n_frames) * hop + nfft / 2) / fs
    return t_frames, f[sel], S

def decimate_for_plot(t, x, max_points=None):
    """Pass plotting data through unchanged by default (max_points=None).
    To thin a long record for display, set max_points to an integer: each
    output bin keeps its local minimum and maximum, preserving the visual
    envelope without aliasing. Computations always use the full-rate data."""
    n = len(x)
    if max_points is None or n <= max_points:
        return np.asarray(t), np.asarray(x)
    bins = max_points // 2
    edge = (n // bins) * bins
    xb = np.asarray(x)[:edge].reshape(bins, -1)
    tb = np.asarray(t)[:edge].reshape(bins, -1)
    tt = np.repeat(tb.mean(axis=1), 2)
    xx = np.empty(2 * bins)
    xx[0::2] = xb.min(axis=1)
    xx[1::2] = xb.max(axis=1)
    tt = np.append(tt, np.asarray(t)[-1])
    xx = np.append(xx, np.asarray(x)[-1])
    return tt, xx

def resample_to_audio(x, fs_in, fs_out=AUDIO_RATE):
    """Linearly interpolate a simulated record onto the audio sample grid so
    it can be played."""
    t_in = np.arange(len(x)) / fs_in
    t_out = np.arange(int(t_in[-1] * fs_out)) / fs_out
    return np.interp(t_out, t_in, x)


---

## C.1 Milling Description

In milling, a rotating tool with defined cutting edges is moved relative to a workpiece to remove material and obtain the desired geometry and dimensions [1]. The tool is mounted in a holder attached to the spindle, and the spindle provides the tool's rotational speed, torque, and power. At minimum, three mutually perpendicular linear axes manipulate the tool-holder-spindle relative to the workpiece, traditionally labeled x, y, and z, with z indicating the tool axis; machines with additional rotational axes provide contouring capability for non-prismatic parts. Milling machines may be manual or computer numerically controlled, and both vertical and horizontal spindle configurations are available.

Cutting tools come in many varieties tailored to peripheral, end, contour, and face milling; for analysis we focus on peripheral and end milling, although the concepts extend to other operations. Endmills are loosely categorized by their free-end geometry as square, ball nose, or bull nose, and the teeth are either ground into the body (typically high-speed steel or sintered carbide) or provided as replaceable inserts clamped to a steel body. The cutting edge is not usually parallel to the tool axis; it is inclined with a helix angle $\gamma$ so the chip is spread over an increased edge length and the cutting pressure is reduced. The chip width $\tilde{b}$ is then related to the axial depth of cut $b$ by

$$\tilde{b} = \frac{b}{\cos\gamma} \tag{4.1}$$

For the straight-teeth analysis of this lesson we take $\gamma = 0$, so the chip width and axial depth coincide.

Under the circular tool path approximation, the instantaneous chip thickness varies with the cutter's rotation angle $\phi$:

$$h(\phi) = f_t \sin(\phi) \tag{4.2}$$

where $f_t$ is the feed per tooth, related to the linear feed $f$ (mm/min), the spindle speed $\Omega$ (rpm), and the tooth count $N_t$ by

$$f_t = \frac{f}{N_t \, \Omega} \tag{4.3}$$

The chip thickness is zero at $\phi = 0$ and 180 deg and reaches its maximum of $f_t$ at 90 deg. In conventional, or up, milling the chip thickness increases as the tooth sweeps through the cut, while in climb, or down, milling it decreases; the falling force toward the finished surface is one reason down milling is often selected for finishing passes. The engagement window depends on the radial depth of cut $a$ and tool radius $r$. For up milling the start angle is $\phi_s = 0$ and the exit angle is

$$\phi_e = \cos^{-1}\!\left(\frac{r - a}{r}\right) \tag{4.4}$$

while for down milling the exit angle is $\phi_e = 180$ deg and the start angle follows from the same geometry. A cut with $a$ equal to half the diameter is called 50 percent radial immersion, and a slot is 100 percent.

<div style="background-color: #e8f4fd; border-left: 5px solid #2196F3; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>Let's Talk About: Nothing Is Constant in Milling</strong><br>
Whereas in turning the chip thickness and chip width are fixed, this is not the case in milling. In a straight slotting cut the chip thickness encountered by each tooth varies continuously as that tooth enters and exits the cut; in pocket milling the radial depth may change as well, and in sculptured surface milling even the axial depth varies. The motion of the rotating tool is simple to visualize, and the exact engagement conditions of the cutting edge can still be surprisingly complicated.
</div>

The cutting force on a tooth follows the turning treatment: proportional to the chip area through the specific force. The tangential and normal components are

$$F_t = k_t \, b \, h(\phi) \tag{4.7}$$

$$F_n = k_n \, b \, h(\phi) \tag{4.8}$$

where $k_t$ and $k_n$ are the tangential and normal cutting force coefficients for the tool-material pair, equivalently expressed through the specific force $K_s = \sqrt{k_t^2 + k_n^2}$ and force angle $\beta = \tan^{-1}(k_t / k_n)$. These components rotate with the tool; measurement and analysis prefer the fixed frame, so they are projected using the cutter angle:

$$F_x = F_t \cos(\phi) + F_n \sin(\phi) \tag{4.9}$$

$$F_y = F_t \sin(\phi) - F_n \cos(\phi) \tag{4.10}$$

When more than one tooth can be engaged, the fixed-frame forces sum over the teeth, with each tooth's angle $\phi_j$ checked against the window $[\phi_s, \phi_e]$ before its contribution is added:

$$F_x = \sum_{j=1}^{N_t} \left[F_t(\phi_j)\cos(\phi_j) + F_n(\phi_j)\sin(\phi_j)\right] \tag{4.15}$$

$$F_y = \sum_{j=1}^{N_t} \left[F_t(\phi_j)\sin(\phi_j) - F_n(\phi_j)\cos(\phi_j)\right] \tag{4.16}$$

The explorer below shows the chip thickness window and the resulting single-revolution force profiles for a set of cut geometries, computed with the Example 4.2 parameters ($k_t = 750$ and $k_n = 250$ N/mm², $b = 5$ mm, $f_t = 0.1$ mm/tooth, four teeth).


In [ ]:
# --- Fig. C.1: chip thickness and forces across cut geometries ---
kt_c, kn_c = 750e6, 250e6            # N/m^2
b_c, ft_c = 5e-3, 0.1e-3             # m, m/tooth
Nt_c = 4

GEOMS = [('25% radial immersion, up', 0.0, 60.0),
         ('50% radial immersion, up', 0.0, 90.0),
         ('100% radial immersion (slot)', 0.0, 180.0),
         ('50% radial immersion, down', 90.0, 180.0),
         ('25% radial immersion, down', 120.0, 180.0)]
START_G = 2   # slider starts at the slot

phi_deg = np.linspace(0, 360, 1441)
phi_rad = np.deg2rad(phi_deg)

fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58],
                    subplot_titles=('Chip thickness for one tooth (Eq. 4.2)',
                                    'One revolution of force (Eqs. 4.15 and 4.16)'))
for gi, (name, ps, pe) in enumerate(GEOMS):
    in_cut = (phi_deg >= ps) & (phi_deg <= pe)
    h = np.where(in_cut, ft_c * np.sin(phi_rad), 0.0)
    Fx = np.zeros_like(phi_deg)
    Fy = np.zeros_like(phi_deg)
    for j in range(Nt_c):
        pj_deg = (phi_deg + j * 360 / Nt_c) % 360
        pj = np.deg2rad(pj_deg)
        eng = (pj_deg >= ps) & (pj_deg <= pe)
        hj = np.where(eng, ft_c * np.sin(pj), 0.0)
        Ft = kt_c * b_c * hj
        Fn = kn_c * b_c * hj
        Fx += Ft * np.cos(pj) + Fn * np.sin(pj)
        Fy += Ft * np.sin(pj) - Fn * np.cos(pj)
    vis = (gi == START_G)
    fig.add_trace(go.Scatter(x=phi_deg, y=h * 1e3, mode='lines',
                             line=dict(color='#1f77b4'), showlegend=False,
                             visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=phi_deg, y=Fx, mode='lines', name='Fx',
                             line=dict(color='#1f77b4'), visible=vis), row=1, col=2)
    fig.add_trace(go.Scatter(x=phi_deg, y=Fy, mode='lines', name='Fy',
                             line=dict(color='#ff7f0e', dash='dash'),
                             visible=vis), row=1, col=2)
    fig.add_trace(go.Scatter(x=phi_deg, y=np.sqrt(Fx**2 + Fy**2), mode='lines',
                             name='resultant',
                             line=dict(color='#2ca02c', dash='dot'),
                             visible=vis), row=1, col=2)

steps = []
for gi, (name, ps, pe) in enumerate(GEOMS):
    visv = [False] * (4 * len(GEOMS))
    visv[4 * gi:4 * gi + 4] = [True] * 4
    steps.append(dict(method='update', label=name, args=[{'visible': visv}]))

fig.update_layout(
    sliders=[dict(active=START_G, currentvalue=dict(prefix='cut geometry: '), steps=steps)],
    title='Fig. C.1 — Chip thickness and cutting force across cut geometries',
    template='simple_white', height=430,
    legend=dict(x=0.98, y=0.98, xanchor='right'),
    margin=dict(l=60, r=20, t=70, b=95))
fig.update_xaxes(title_text='phi (deg)', row=1, col=1)
fig.update_xaxes(title_text='phi (deg)', row=1, col=2)
fig.update_yaxes(title_text='h (mm)', row=1, col=1)
fig.update_yaxes(title_text='force (N)', row=1, col=2)
fig.show()


Two observations from the explorer carry forward. Down milling mirrors up milling in time, with the force falling toward the finished surface, and the x direction force changes sign between the two. And for the slot, once teeth are engaged continuously the summed force approaches a steadier profile; slotting with an even tooth count above two produces constant total force.

### C.1.1 Tooth Passing Frequency

The forces above are plotted against angle; the spindle speed converts angle to time, and the force record becomes a train of periodic pulses, exactly the signal class of Sect. B.7. Its fundamental frequency is the *tooth passing frequency*:

$$f_{tooth} = \frac{\Omega \, N_t}{60} \tag{4.14}$$

with $\Omega$ in rpm. The spectrum of the cutting force therefore contains $f_{tooth}$ and its integer harmonics, plus DC content from the nonzero mean, and the relative harmonic magnitudes depend on the sharpness of the force pulses: lower radial immersion means more impulse-like forces and stronger high harmonics. Because the forces excite the structure, a stable cut is forced vibration at these frequencies, and changing the spindle speed moves the entire excitation family. The cell below lets you hear it.


In [ ]:
# --- The tooth passing rhythm. Adjust and rerun. ---
omega_rpm = 7500     # spindle speed (rpm). Try changing these!
Nt = 4               # number of teeth

f_tooth = omega_rpm * Nt / 60
print(f'Eq. 4.14: f_tooth = {omega_rpm} x {Nt} / 60 = {f_tooth:.0f} Hz '
      f'(spindle rotation at {omega_rpm / 60:.1f} Hz)')

dur = 1.5
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
phase = (t * f_tooth) % 1.0
pulses = np.where(phase < 0.12, np.sin(np.pi * phase / 0.12), 0.0)
display(Audio(fade(0.6 * (pulses - pulses.mean())), rate=AUDIO_RATE))


<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
If the radial depth of cut is very small, there are long periods when no tooth is cutting at all and the force record is a series of very short impacts, rich in high harmonics. At larger radial depths the number of simultaneously engaged teeth may be constant or may alternate, depending on the geometry and tooth spacing. Reading a machining spectrum begins with Eq. 4.14: compute $f_{tooth}$ from the programmed spindle speed and tooth count, and every line of the healthy family should sit at an integer multiple of it.
</div>

▸ **Key terms: milling, up milling, down milling, feed per tooth, radial immersion, chip thickness, tooth passing frequency**


---

## C.2 The Sound of a Stable Cut

Part B ended with a prediction: a repeating cutting force must produce harmonics at multiples of its fundamental frequency. A real machine adds two more ingredients. The idle machine has a baseline of broadband noise and motor-related tones that exists before any cutting. And a real cutter never runs perfectly true: *runout*, the small offset between the tool's geometric and rotational centers, makes the teeth cut slightly unequal chips, which adds content at the once-per-revolution frequency $\Omega/60$ and its harmonics, spaced $N_t$ times more finely than the tooth passing family [1]. The explorer below assembles a synthetic stable cut one ingredient at a time, for the Example 4.11 conditions (7,500 rpm, four teeth, so $f_{tooth} = 500$ Hz and the runout spacing is 125 Hz).


In [ ]:
# --- Fig. C.2: a stable cut assembled ingredient by ingredient ---
dur = 2.0
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
rng = np.random.default_rng(26)

f_rev = 7500 / 60          # 125 Hz once-per-revolution
f_tp = 500.0               # tooth passing fundamental

baseline = 0.06 * rng.standard_normal(len(t))
for f_m in (60, 120, 240):                       # motor-related tones
    baseline += 0.03 * np.sin(2 * np.pi * f_m * t)

runout = np.zeros_like(t)
for m in range(1, 8):
    if m % 4:                                    # skip multiples that overlap f_tooth
        runout += (0.12 / m) * np.sin(2 * np.pi * m * f_rev * t)

tooth = np.zeros_like(t)
for m in range(1, 9):
    tooth += (0.8 / m) * np.sin(2 * np.pi * m * f_tp * t)

STAGES = [('idle machine', baseline),
          ('idle + spindle runout', baseline + runout),
          ('idle + runout + tooth passing', baseline + runout + tooth)]

fig = go.Figure()
for si, (name, sig) in enumerate(STAGES):
    f_sp, a_sp = compute_spectrum(sig, AUDIO_RATE, fmax=2600)
    fig.add_trace(go.Scatter(x=f_sp, y=a_sp, mode='lines',
                             line=dict(color='#1f77b4'),
                             visible=(si == 2), showlegend=False))
steps = []
for si, (name, sig) in enumerate(STAGES):
    vis = [False] * len(STAGES)
    vis[si] = True
    steps.append(dict(method='update', label=name, args=[{'visible': vis}]))
fig.update_layout(
    sliders=[dict(active=2, currentvalue=dict(prefix='ingredients: '), steps=steps)],
    title='Fig. C.2 — The spectrum of a stable cut, ingredient by ingredient',
    template='simple_white', height=400,
    xaxis_title='frequency (Hz)', yaxis_title='amplitude (normalized)',
    yaxis_range=[0, 1.05],
    margin=dict(l=60, r=20, t=60, b=95))
fig.show()

for name, sig in STAGES:
    print(name + ':')
    display(Audio(fade(0.6 * sig / np.max(np.abs(sig))), rate=AUDIO_RATE))


Every line in the final spectrum has a name. The dense low family spaced at 125 Hz is the spindle's once-per-revolution content from runout, the tall family at 500 Hz and multiples is the tooth passing family, and everything else is the idle baseline. This is the complete sound of a healthy cut, and the practical skill is the negative statement it enables: any strong line that belongs to none of these families is unexplained, and Sect. C.7 gives the strongest candidate an explanation.

▸ **Key terms: runout, once-per-revolution frequency**


---

## C.3 Regenerative Chatter in Milling

Now remove the rigid-tool assumption. The cutting force deflects the tool, and if the tool vibrates while removing material, the vibration is imprinted on the workpiece surface as a wavy profile. In milling, the time-delayed surface regeneration step occurs from tooth to tooth: the wavy surface left behind by tooth 1 is removed by tooth 2, one tooth period $\tau = 60/(\Omega N_t)$ later [1]. The instantaneous chip thickness therefore depends on both the current vibration and the surface left by the previous tooth, so vibration feeds back into force, which feeds back into vibration. This closed loop is *regeneration*, and it makes chatter possible in milling exactly as in turning.

The chip thickness is measured along the surface normal, whose direction rotates with the cutter angle. Projecting the x and y vibrations onto the normal,

$$n = x \sin(\phi) - y \cos(\phi) \tag{4.21}$$

with positive $n$ out of the cut, the instantaneous chip thickness becomes

$$h(t) = f_t \sin(\phi) + n(t - \tau) - n(t) \tag{4.22}$$

where the $n(t-\tau)$ term is the previous tooth's contribution along the current normal. Whether the loop feeds or starves itself depends on the phase between the wave left by one tooth and the vibration of the next. When the two are in phase, the chip thickness variation is no worse than the cycloidal path alone produces, and the cut remains a forced vibration; matching the tooth passing frequency to the natural frequency encourages exactly this favorable phasing. When the passes are out of phase, the chip thickness varies strongly, the force varies with it, and the vibration can grow tooth by tooth: self-excited vibration, the third kind from Sect. A.2. The explorer below shows the two surfaces and the chip between them as the phase $\varepsilon$ varies.


In [ ]:
# --- Fig. C.3: two successive tooth passes and the chip between them ---
u = np.linspace(0, 4 * np.pi, 500)
amp = 1.0
EPS_STEPS = list(range(0, 361, 30))
START_E = 6   # slider starts at 180 deg

fig = go.Figure()
prev = amp * np.sin(u)
for ei, eps in enumerate(EPS_STEPS):
    curr = amp * np.sin(u + np.deg2rad(eps)) - 3.0     # offset: one feed below
    vis = (ei == START_E)
    fig.add_trace(go.Scatter(x=u, y=prev, mode='lines',
                             name='surface left by previous tooth',
                             line=dict(color='#1f77b4'), visible=vis))
    fig.add_trace(go.Scatter(x=u, y=curr, mode='lines',
                             name='path of current tooth',
                             line=dict(color='#ff7f0e', dash='dash'),
                             visible=vis))
    fig.add_trace(go.Scatter(x=np.concatenate([u, u[::-1]]),
                             y=np.concatenate([prev, curr[::-1]]),
                             fill='toself', fillcolor='rgba(44,160,44,0.18)',
                             line=dict(width=0), name='chip', visible=vis,
                             hoverinfo='skip'))

steps = []
for ei, eps in enumerate(EPS_STEPS):
    vis = [False] * (3 * len(EPS_STEPS))
    vis[3 * ei:3 * ei + 3] = [True] * 3
    steps.append(dict(method='update', label=f'{eps}', args=[{'visible': vis}]))
fig.update_layout(
    sliders=[dict(active=START_E,
                  currentvalue=dict(prefix='phase between passes (deg) = '),
                  steps=steps)],
    title='Fig. C.3 — The chip between successive tooth passes',
    template='simple_white', height=400,
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
    margin=dict(l=20, r=20, t=60, b=95))
fig.show()


At 0 or 360 deg the two waves nest and the chip has nearly constant thickness: the vibration exists, and it does no regenerative harm. Near 180 deg the waves oppose and the chip thickness swings between nearly zero and nearly double, so the force swings with it and pumps the vibration. Whether that pumping overcomes the system's damping depends on the axial depth $b$, which scales the force, and the next section computes the boundary.

▸ **Key terms: regeneration, tooth period, tooth-to-tooth phase, chatter**


---

## C.4 Stability Lobe Diagrams

The boundary between stable machining and chatter, drawn as the limiting axial depth $b_{lim}$ against spindle speed $\Omega$, is the *stability lobe diagram*. This section computes it twice, by two analytical solutions, each stated with its central assumption. Both linearize the same physics; they differ in how the rotating, time-varying cut geometry is reduced to a constant one. Each is summarized by its governing equations below; the implementations sit in collapsed cells, one expand-click away, and both boundaries are drawn together on one figure.

### C.4.1 Analytical Solution 1: The Average Tooth Angle Approach

The first approach evaluates the cut geometry at a single representative angle, the average angle of a tooth in the cut, $\phi_{ave} = (\phi_s + \phi_e)/2$. The x and y frequency response functions are combined into one *oriented FRF* through directional orientation factors obtained by projecting the cutting force onto each mode direction and that result onto the average surface normal [1]:

$$FRF_{orient} = \mu_x \, FRF_x + \mu_y \, FRF_y \tag{4.27}$$

For a slot, $\phi_{ave} = 90$ deg and the projections give $\mu_x = \cos\beta$ and $\mu_y = 0$; the general projections are computed in the code below. The stability boundary is then a mapping of the real part of the oriented FRF, valid where it is negative:

$$b_{lim} = \frac{-1}{2 \, K_s \, \mathrm{Re}\!\left[FRF_{orient}\right] N_t^*} \tag{4.23}$$

where $N_t^* = N_t(\phi_e - \phi_s)/360$ is the average number of teeth in the cut. Each valid chatter frequency $f_c$ maps to one spindle speed per lobe number $N$ through the tooth-to-tooth phase $\varepsilon$, with the best speeds at $\varepsilon = 2\pi$:

$$\Omega = \frac{60 \, f_c}{N_t \left(N + \varepsilon / 2\pi\right)} \tag{4.24}$$


In [ ]:
# --- Analytical solution 1: average tooth angle (Tlusty) stability lobes ---
def frf_sdof(f, fn, k, zeta):
    r = f / fn
    return (1 / k) / ((1 - r**2) + 1j * 2 * zeta * r)

def tlusty_lobes(fn, k, zeta, Ks, beta_deg, Nt, phis_deg, phie_deg,
                 lobes=range(0, 6), n_f=4000):
    '''Average tooth angle stability lobes per Sect. 4.3.1 and 4.3.2 of [1].
    Assumption: the rotating cut geometry is evaluated at the single average
    tooth angle. Returns a list of (omega_rpm, blim_mm) curves, one per lobe.'''
    beta = np.deg2rad(beta_deg)
    phi_ave = np.deg2rad(0.5 * (phis_deg + phie_deg))
    theta_n = phi_ave - np.pi / 2                     # surface normal from +x
    mu_x = np.cos(beta + theta_n) * np.cos(theta_n)
    mu_y = np.sin(beta + theta_n) * np.sin(theta_n)
    Nt_star = Nt * (phie_deg - phis_deg) / 360

    fc = np.linspace(fn * 1.0001, fn * 4, n_f)
    FRFor = mu_x * frf_sdof(fc, fn, k, zeta) + mu_y * frf_sdof(fc, fn, k, zeta)
    R, I = FRFor.real, FRFor.imag
    valid = R < 0
    fc, R, I = fc[valid], R[valid], I[valid]

    blim = -1.0 / (2 * Ks * R * Nt_star) * 1e3        # m -> mm
    eps = 2 * np.arctan2(-R, I)                       # tooth-to-tooth phase (rad)
    curves = []
    for N in lobes:
        omega = 60 * fc / (Nt * (N + eps / (2 * np.pi)))    # Eq. 4.24
        curves.append((omega, blim, fc))
    return curves

print('tlusty_lobes defined: the average tooth angle analytical solution.')


### C.4.2 Analytical Solution 2: The Fourier Series (Zero-Order) Approach

The second approach, due to Altintas and Budak [2] and developed in Sect. 4.3.3 of [1], keeps the full time-varying directional coefficients of the milling equations and expands them in a Fourier series over the tooth period; truncating the series at the average term, the zero-order approximation, removes the time dependence. The averaged directional matrix is

$$[A_0] = \frac{N_t}{2\pi}\begin{bmatrix} \alpha_{xx} & \alpha_{xy} \\ \alpha_{yx} & \alpha_{yy} \end{bmatrix} \tag{4.44}$$

with, for example, the xx entry given by

$$\alpha_{xx} = \frac{1}{2}\left[\cos 2\phi - 2 K_r \phi + K_r \sin 2\phi\right]_{\phi_s}^{\phi_e} \tag{4.45}$$

and the companion $\alpha_{xy}$, $\alpha_{yx}$, and $\alpha_{yy}$ integrals written out in the collapsed cell. Stability of the resulting time-invariant loop is an eigenvalue problem: the characteristic equation of the closed loop (Eq. 4.50 of [1]) is quadratic in the eigenvalue $\Lambda$,

$$a_0 \Lambda^2 + a_1 \Lambda + 1 = 0, \qquad a_0 = FRF_{xx}\,FRF_{yy}\left(\alpha_{xx}\alpha_{yy} - \alpha_{xy}\alpha_{yx}\right), \qquad a_1 = \alpha_{xx}\,FRF_{xx} + \alpha_{yy}\,FRF_{yy}$$

and each root $\Lambda = \Lambda_{Re} + i\Lambda_{Im}$ gives a candidate limiting depth and phase (Sect. 4.3.3 of [1]):

$$b_{lim} = -\frac{2\pi \Lambda_{Re}}{N_t k_t}\left(1 + \kappa^2\right), \qquad \kappa = \frac{\Lambda_{Im}}{\Lambda_{Re}}, \qquad \psi = \tan^{-1}\kappa, \qquad \varepsilon = \pi - 2\psi, \qquad \Omega = \frac{60 f_c}{N_t\left(\varepsilon/2\pi + j\right)}$$

for lobe numbers $j = 0, 1, 2, \ldots$ Only positive depths are physical and retained, and the composite boundary is the minimum over both eigenvalues at each spindle speed, exactly as Fig. 4.34 of [1] plots it. One detail matters here and Sect. 4.3.3 of [1] states it explicitly: the full chatter frequency range is scanned, including frequencies below the natural frequency, because the coupled two direction system can chatter there. Both approaches share the accuracy caveat stated in Chap. 4 of [1]: as radial immersion falls and the force becomes impulse-like, averaging (by either route) degrades. The collapsed cells below carry the implementing lines; expand them whenever wanted.


In [ ]:
# --- Analytical solution 2: Fourier series (zero-order) stability lobes ---
def zoa_lobes(fn, k, zeta, kt, kn, Nt, phis_deg, phie_deg,
              lobes=range(0, 13), n_f=40000):
    '''Fourier series (zero-order) stability lobes per Sect. 4.3.3 of [1],
    following Altintas and Budak [2]. Only the average term of the Fourier
    series of the time varying directional coefficients is retained (Eqs.
    4.44 and 4.45). At each candidate chatter frequency the quadratic
    characteristic equation a0*Lam^2 + a1*Lam + 1 = 0 gives two eigenvalues
    (Eq. 4.53); each eigenvalue gives a candidate limiting depth (Eq. 4.58),
    and only positive depths are physical and retained; the regenerative
    phase then assigns one spindle speed per lobe number j (Eq. 4.59). Per
    [1], the full chatter frequency range is scanned, including frequencies
    below fn, where the coupled two direction system can also chatter.
    Returns a list of (omega_rpm, blim_mm, fc_hz) branches.'''
    Kr = kn / kt

    def alpha_anti(p):
        # antiderivatives of the Eq. 4.45 integrands, evaluated at angle p (rad)
        return np.array([0.5 * ( np.cos(2*p) - 2*Kr*p + Kr*np.sin(2*p)),   # xx
                         0.5 * (-np.sin(2*p) - 2*p   + Kr*np.cos(2*p)),    # xy
                         0.5 * (-np.sin(2*p) + 2*p   + Kr*np.cos(2*p)),    # yx
                         0.5 * (-np.cos(2*p) - 2*Kr*p - Kr*np.sin(2*p))])  # yy
    axx, axy, ayx, ayy = (alpha_anti(np.deg2rad(phie_deg))
                          - alpha_anti(np.deg2rad(phis_deg)))

    fc = np.linspace(30.0, fn * 8, n_f)               # full frequency range
    Phix = frf_sdof(fc, fn, k, zeta)
    Phiy = frf_sdof(fc, fn, k, zeta)
    a0 = Phix * Phiy * (axx * ayy - axy * ayx)
    a1 = axx * Phix + ayy * Phiy
    disc = np.sqrt(a1**2 - 4 * a0 + 0j)

    curves = []
    for Lam in ((-a1 + disc) / (2 * a0), (-a1 - disc) / (2 * a0)):
        LR, LI = Lam.real, Lam.imag
        with np.errstate(all='ignore'):
            kappa = LI / LR
            blim = -(2 * np.pi * LR / (Nt * kt)) * (1 + kappa**2) * 1e3
        keep = np.isfinite(blim) & (blim > 0)         # positive depths only
        edges = np.where(np.diff(keep))[0] + 1        # contiguous valid runs
        for run in np.split(np.arange(len(fc)), edges):
            if len(run) < 2 or not keep[run[0]]:
                continue
            psi = np.arctan(kappa[run])
            eps = np.pi - 2 * psi
            for j in lobes:
                om = 60 * fc[run] / (Nt * (eps / (2 * np.pi) + j))
                curves.append((om, blim[run], fc[run]))
    return curves

def lower_envelope(curves, om_grid):
    '''Composite stability boundary: the minimum limiting depth over all lobe
    branches at each spindle speed, as [1] plots in its Fig. 4.34. Each
    branch is a continuous curve traced by the chatter frequency, so the
    minimum is taken over linear interpolations along each branch, split
    into monotone pieces wherever a branch folds back in spindle speed.
    Interpolating along the curve matters on the branch tails, where a small
    chatter frequency step sweeps a large speed range; collecting isolated
    points into speed bins under-samples those tails and produces jagged,
    grid-dependent artifacts at high spindle speed.'''
    env = np.full(len(om_grid), np.inf)
    for om, bl, fc in curves:
        d = np.sign(np.diff(om))
        brk = np.where(d[1:] != d[:-1])[0] + 1
        for p in np.split(np.arange(len(om)), brk + 1):
            if len(p) < 2:
                continue
            o, b = om[p], bl[p]
            if o[0] > o[-1]:
                o, b = o[::-1], b[::-1]
            env = np.minimum(env, np.interp(om_grid, o, b,
                                            left=np.inf, right=np.inf))
    return env

print('zoa_lobes and lower_envelope defined: the Fourier series solution.')


With both solutions in hand, we draw them together for the Example 4.11 system (fn = 500 Hz, k = 8 MN/m, ζ = 0.02 in both directions, four teeth, slotting, kt = 695 and kn = 281 N/mm², so Ks = 750 N/mm² and β = 68 deg), and mark the four operating points the simulation of Sect. C.5 will test.


In [ ]:
# --- Fig. C.4: both analytical boundaries with the Example 4.11 cases ---
PAR = dict(fn=500.0, k=8e6, zeta=0.02, Nt=4, phis_deg=0.0, phie_deg=180.0)
KT, KN = 695e6, 281e6
KS = np.sqrt(KT**2 + KN**2)
BETA = np.rad2deg(np.arctan2(KT, KN))
CASES = [('1', 7500, 3.0, 'stable'), ('2', 7500, 5.0, 'unstable'),
         ('3', 5000, 0.1, 'stable'), ('4', 5000, 0.5, 'unstable')]

tl = tlusty_lobes(PAR['fn'], PAR['k'], PAR['zeta'], KS, BETA,
                  PAR['Nt'], PAR['phis_deg'], PAR['phie_deg'])
zo = zoa_lobes(PAR['fn'], PAR['k'], PAR['zeta'], KT, KN,
               PAR['Nt'], PAR['phis_deg'], PAR['phie_deg'])
zo_om = np.linspace(2000, 16000, 561)
zo_bl = lower_envelope(zo, zo_om)

fig = go.Figure()
for i, (om, bl, fc) in enumerate(tl):
    fig.add_trace(go.Scatter(x=om, y=bl, mode='lines',
                             line=dict(color='#1f77b4'),
                             name='average tooth angle', showlegend=(i == 0),
                             legendgroup='tl'))
fig.add_trace(go.Scatter(x=zo_om, y=zo_bl, mode='lines',
                         line=dict(color='#ff7f0e', dash='dash'),
                         name='Fourier series (zero-order)'))
for label, om, b, verdict in CASES:
    color = '#2ca02c' if verdict == 'stable' else '#d62728'
    fig.add_trace(go.Scatter(x=[om], y=[b], mode='markers+text',
                             marker=dict(color=color, size=11,
                                         symbol='circle' if verdict == 'stable' else 'x'),
                             text=[label], textposition='top center',
                             showlegend=False))
fig.update_layout(title='Fig. C.4 — Stability lobes by two analytical solutions',
                  template='simple_white', height=470,
                  xaxis=dict(title='spindle speed (rpm)', range=[2000, 16000]),
                  yaxis=dict(title='b_lim (mm)', range=[0, 8]),
                  legend=dict(x=0.02, y=0.98),
                  margin=dict(l=60, r=20, t=60, b=50))
fig.show()

def blim_at(curves, omega_rpm):
    best = np.inf
    for om, bl, fc in curves:
        sel = (om > omega_rpm - 60) & (om < omega_rpm + 60)
        if sel.any():
            best = min(best, np.min(bl[sel]))
    return best

print('Boundary near each case, and the verdict each method implies:')
for label, om, b, verdict in CASES:
    bt = blim_at(tl, om)
    bz = float(np.interp(om, zo_om, zo_bl))
    vt = 'stable' if b < bt else 'unstable'
    vz = 'stable' if b < bz else 'unstable'
    print(f'case {label} ({om} rpm, {b} mm): '
          f'average tooth angle {bt:5.2f} mm implies {vt}; '
          f'zero-order {bz:5.2f} mm implies {vz}; Example 4.11 of [1] reports {verdict}')


The two solutions disagree, and the disagreement is the lesson of this section. For slotting, the average tooth angle is $\phi_{ave} = 90$ deg, which places the average surface normal parallel to the x direction. The projection of the y direction onto that normal is then zero, $\mu_y = 0$, and the average tooth angle approach discards the y direction dynamics entirely; its boundary reflects the x direction alone and reaches nearly 9 mm at the 7,500 rpm pocket. The zero-order approach retains both directions through the off-diagonal terms of Eq. 4.44, and its boundary is markedly lower, about 3.7 mm at 7,500 rpm and about 0.23 mm at 5,000 rpm, with wide low bands on both sides of the pocket. Much of the reduction comes from chatter frequencies below the natural frequency, which exist only because the two directions are coupled. Past the last pocket the boundary holds near 0.23 mm across the rest of this diagram, so raising the spindle speed alone does not raise the limiting depth for this system; the deeper stable regions are the pockets. The printout above judges the four cases against each curve: the average tooth angle boundary calls cases 2 and 4 stable, while the zero-order boundary reproduces all four Example 4.11 verdicts. Sect. 4.4 of [1] states the reason plainly: the disagreement for cases 2 and 4 is due to the orthogonality between the average surface normal and the y direction in slotting, which forces $\mu_y = 0$ and neglects the contribution of the y direction dynamics. Averaging the geometry at a single angle is a modeling assumption, and here it costs roughly a factor of two in the predicted limit. Sect. C.5 applies a time domain simulation, which makes no averaging assumption, to decide between them.

Both boundaries keep the lobed shape that makes the diagram useful in practice: near spindle speeds where the tooth passing frequency divides evenly into the natural frequency, the stable pockets reach several times the depth available elsewhere, and case 1 sits inside such a pocket. The floor of the whole diagram is set by damping. The explorer below recomputes both boundaries at three damping ratios; drag between them and watch the valleys rise and fall together.


In [ ]:
# --- Fig. C.5: the stability boundary as damping varies ---
ZETAS = [0.01, 0.02, 0.05]
START_Z = 1

fig = go.Figure()
n_per = []
for zi, zval in enumerate(ZETAS):
    tl_z = tlusty_lobes(PAR['fn'], PAR['k'], zval, KS, BETA,
                        PAR['Nt'], PAR['phis_deg'], PAR['phie_deg'])
    zo_bl_z = lower_envelope(zoa_lobes(PAR['fn'], PAR['k'], zval, KT, KN,
                                       PAR['Nt'], PAR['phis_deg'],
                                       PAR['phie_deg']), zo_om)
    vis = (zi == START_Z)
    for i, (om, bl, fc) in enumerate(tl_z):
        fig.add_trace(go.Scatter(x=om, y=bl, mode='lines',
                                 line=dict(color='#1f77b4'),
                                 name='average tooth angle',
                                 showlegend=(i == 0), legendgroup='tl',
                                 visible=vis))
    fig.add_trace(go.Scatter(x=zo_om, y=zo_bl_z, mode='lines',
                             line=dict(color='#ff7f0e', dash='dash'),
                             name='Fourier series (zero-order)',
                             legendgroup='zo', visible=vis))
    n_per.append(len(tl_z) + 1)

steps = []
total = sum(n_per)
for zi, zval in enumerate(ZETAS):
    vis = [False] * total
    ofs = sum(n_per[:zi])
    vis[ofs:ofs + n_per[zi]] = [True] * n_per[zi]
    steps.append(dict(method='update', label=f'{zval:.2f}', args=[{'visible': vis}]))
fig.update_layout(
    sliders=[dict(active=START_Z, currentvalue=dict(prefix='zeta = '), steps=steps)],
    title='Fig. C.5 — Damping sets the floor of the stability lobe diagram',
    template='simple_white', height=440,
    xaxis=dict(title='spindle speed (rpm)', range=[2000, 16000]),
    yaxis=dict(title='b_lim (mm)', range=[0, 10]),
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=60, r=20, t=60, b=95))
fig.show()


▸ **Key terms: stability lobe diagram, limiting axial depth, oriented FRF, directional orientation factors, average tooth angle approach, zero-order approximation**


---

## C.5 The Milling Time Domain Simulation

The analytical solutions are fast and rest on averaging assumptions. The time domain simulation of Sect. 4.4 of [1] makes neither approximation: it marches the cut forward in small time steps, keeping the full rotating geometry, the tooth-by-tooth surface regeneration of Eq. 4.22, and the nonlinearity of a tooth that vibrates out of the cut entirely. This section is a faithful Python port of p_4_10_1.m from [1], kept in a collapsed cell and expandable whenever the line-by-line detail is wanted.

The strategy divides one revolution into `steps_rev` equal angle increments $d\phi = 360/\texttt{steps\_rev}$ deg, with time step $dt = 60/(\texttt{steps\_rev} \cdot \Omega)$ s, and requires `steps_rev` to be a multiple of $N_t$ so the teeth land exactly on grid angles. A vector indexed by grid angle stores the surface left by the previous tooth at each orientation. Every time step performs three activities [1]: rotate the cutter by one increment; for each tooth inside the window $[\phi_s, \phi_e]$, form the chip thickness from Eq. 4.22 using the stored surface and the current normal-direction vibration (Eq. 4.21), compute the forces by Eqs. 4.7 to 4.10 with the stored surface updated to the current vibration, or, if the chip thickness is negative, set the forces to zero and add the feed contribution to the stored surface because the tooth has vibrated out of the cut; and integrate the x and y equations of motion one step. The function below is the complete simulation.


In [ ]:
# --- The milling time domain simulation (Python equivalent of p_4_10_1.m) ---
def mill_tds(omega_rpm, b_mm, Nt=4, ft_mm=0.15, phis_deg=0.0, phie_deg=180.0,
             fn_x=500.0, k_x=8e6, zeta_x=0.02,
             fn_y=500.0, k_y=8e6, zeta_y=0.02,
             kt=695e6, kn=281e6, steps_rev=652, revolutions=20):
    """Milling time domain simulation with straight teeth, per Sect. 4.4 of
    [1]. Returns a dict with time (s), forces Fx, Fy (N), displacements
    x, y (m), the number of steps per tooth period, and the sample rate (Hz).
    steps_rev is rounded to the nearest multiple of Nt as in p_4_10_1.m."""
    steps_rev = int(round(steps_rev / Nt) * Nt)
    dphi = 2 * np.pi / steps_rev
    dt = 60.0 / (steps_rev * omega_rpm)
    steps = steps_rev * revolutions
    b = b_mm * 1e-3
    ft = ft_mm * 1e-3
    ps, pe = np.deg2rad(phis_deg), np.deg2rad(phie_deg)

    phi = np.arange(steps_rev) * dphi
    sin_p, cos_p = np.sin(phi), np.cos(phi)
    in_cut = (phi >= ps) & (phi <= pe)
    teeth = (np.arange(Nt) * (steps_rev // Nt)).astype(int)
    surf = np.zeros(steps_rev)

    wnx, wny = 2 * np.pi * fn_x, 2 * np.pi * fn_y
    mx, my = k_x / wnx**2, k_y / wny**2
    cx, cy = 2 * zeta_x * np.sqrt(mx * k_x), 2 * zeta_y * np.sqrt(my * k_y)

    Fx = np.zeros(steps); Fy = np.zeros(steps)
    xh = np.zeros(steps); yh = np.zeros(steps)
    x = y = vx = vy = 0.0

    for cnt in range(steps):
        teeth = (teeth + 1) % steps_rev
        fx_s = fy_s = 0.0
        for j in range(Nt):
            idx = teeth[j]
            if not in_cut[idx]:
                continue
            s, c = sin_p[idx], cos_p[idx]
            n = x * s - y * c                       # Eq. 4.21
            h = ft * s + surf[idx] - n              # Eq. 4.22
            if h < 0:                               # tooth vibrated out of the cut
                surf[idx] += ft * s
                continue
            Ft = kt * b * h                         # Eq. 4.7
            Fn = kn * b * h                         # Eq. 4.8
            surf[idx] = n
            fx_s += Ft * c + Fn * s                 # Eq. 4.9
            fy_s += Ft * s - Fn * c                 # Eq. 4.10
        Fx[cnt], Fy[cnt] = fx_s, fy_s
        vx += dt * (fx_s - cx * vx - k_x * x) / mx  # Euler integration, x
        x += dt * vx
        vy += dt * (fy_s - cy * vy - k_y * y) / my  # Euler integration, y
        y += dt * vy
        xh[cnt], yh[cnt] = x, y

    return dict(t=np.arange(steps) * dt, Fx=Fx, Fy=Fy, x=xh, y=yh,
                steps_tooth=steps_rev // Nt, fs=1.0 / dt, steps_rev=steps_rev)

def chatter_metric(sim, threshold_um=1.0):
    """Periodic sampling stability metric per Sect. 4.4.5 of [1]. The x
    displacement is sampled once per tooth passage over the second half of
    the record, after the startup transient has decayed, and M is the
    mean absolute difference between
    successive samples, in micrometers. Stable cuts repeat at the tooth
    period, so M sits near zero; chatter does not repeat and M reaches tens
    to hundreds of micrometers ([1] demonstrates 0.06 versus 45.56 for its
    Example 4.13). Cuts with M below one micrometer are classified stable.
    Returns (M_um, verdict)."""
    x = sim['x']
    start = len(x) // 2                     # after the startup transient
    xs = x[np.arange(start, len(x), sim['steps_tooth'])] * 1e6
    M = float(np.sum(np.abs(np.diff(xs))) / len(xs))
    return M, ('stable' if M < threshold_um else 'chatter')

print('mill_tds and chatter_metric defined.')


The parameter panel below runs the simulation. Every quantity is exposed; the defaults are Example 4.11 of [1] exactly, and the `PRESET` line selects among the four Example 4.11 cases. Change any value and rerun. The printout applies the metric to whatever parameters are set; the figure below runs all four cases and shows the x direction displacement of each with the sampled points marked. It applies the periodic sampling metric of Sect. 4.4.5 of [1]: the displacement is sampled once per tooth passage over the second half of the record, after the startup transient has decayed, and $M$ is the mean absolute difference between successive samples, in micrometers. For a stable cut the samples repeat and $M$ sits near zero; for chatter they do not repeat and $M$ reaches tens to hundreds of micrometers (Example 4.13 of [1] demonstrates 0.06 against 45.56). A dividing line of one micrometer classifies every cut in this Part cleanly. Slotting with an even number of teeth makes the stable side especially quiet: Sect. C.1 showed the total force is constant there, so a stable cut carries almost no vibration and its samples are nearly identical.


In [ ]:
# --- Run the simulation. Adjust the parameters and rerun. ---
PRESETS = {'1': dict(omega_rpm=7500, b_mm=3.0),      # stable, per [1]
           '2': dict(omega_rpm=7500, b_mm=5.0),      # unstable
           '3': dict(omega_rpm=5000, b_mm=0.1),      # stable
           '4': dict(omega_rpm=5000, b_mm=0.5)}      # unstable
PRESET = '2'          # pick a case, or set the parameters directly below

params = dict(
    omega_rpm=7500,   # spindle speed (rpm)
    b_mm=5.0,         # axial depth of cut (mm)
    Nt=4,             # number of teeth
    ft_mm=0.15,       # feed per tooth (mm)
    phis_deg=0.0,     # cut start angle (deg)
    phie_deg=180.0,   # cut exit angle (deg): slotting
    fn_x=500.0, k_x=8e6, zeta_x=0.02,     # x direction dynamics
    fn_y=500.0, k_y=8e6, zeta_y=0.02,     # y direction dynamics
    kt=695e6, kn=281e6,                   # cutting force coefficients (N/m^2)
    steps_rev=652, revolutions=20)
if PRESET in PRESETS:
    params.update(PRESETS[PRESET])

sim = mill_tds(**params)
M, verdict = chatter_metric(sim)
f_tooth = params['omega_rpm'] * params['Nt'] / 60
print(f"Case: {params['omega_rpm']} rpm, {params['b_mm']} mm, "
      f"f_tooth = {f_tooth:.0f} Hz.")
print(f"Periodic sampling metric M = {M:.2f} µm -> {verdict}.")


In [ ]:
# --- Fig. C.6: the four Example 4.11 cases ---
case_results = []
for lab in ('1', '2', '3', '4'):
    p4 = dict(params)
    p4.update(PRESETS[lab])
    p4['revolutions'] = 30      # a little longer, so the sampled half is easy to see
    s4 = mill_tds(**p4)
    M4, v4 = chatter_metric(s4)
    case_results.append((lab, p4, s4, M4, v4))

fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.09,
                    vertical_spacing=0.17,
                    subplot_titles=[f"case {lab}: {p4['omega_rpm']:,} rpm, "
                                    f"{p4['b_mm']} mm ({v4}, M = {M4:.2f} µm)"
                                    for lab, p4, s4, M4, v4 in case_results])
for i_c, (lab, p4, s4, M4, v4) in enumerate(case_results):
    r_, c_ = i_c // 2 + 1, i_c % 2 + 1
    tt, xx = decimate_for_plot(s4['t'], s4['x'] * 1e6, max_points=3000)
    fig.add_trace(go.Scatter(x=tt, y=xx, mode='lines',
                             line=dict(color='#1f77b4', width=0.8),
                             showlegend=False), row=r_, col=c_)
    s0 = len(s4['t']) // 2
    si = np.arange(s0, len(s4['t']), s4['steps_tooth'])
    si = si[s4['t'][si] <= tt[-1]]      # samples never extend past the line
    fig.add_trace(go.Scatter(x=s4['t'][si], y=s4['x'][si] * 1e6,
                             mode='markers',
                             marker=dict(color='#d62728', size=4),
                             name='once-per-tooth samples',
                             showlegend=(i_c == 0)), row=r_, col=c_)
    fig.update_xaxes(title_text='t (s)', row=r_, col=c_)
    fig.update_yaxes(title_text='x (µm)', row=r_, col=c_)
fig.update_layout(title='Fig. C.6 — Time domain simulation of the four Example 4.11 cases',
                  template='simple_white', height=560,
                  legend=dict(x=0.0, y=1.12, orientation='h'),
                  margin=dict(l=60, r=20, t=95, b=50))
fig.show()


The four panels land exactly as Example 4.11 of [1] reports: cases 1 and 3 stable and 2 and 4 in chatter, with the once-per-tooth samples holding still for the former and wandering for the latter. Compare each position against the boundaries of Fig. C.4. This matches the zero-order boundary and resolves the Sect. C.4 disagreement in its favor. Because this Part is about sound, the cell below sonifies all four cases: each is rerun long enough to hear, the displacement record becomes audio, and the four clips share one loudness scale so they carry the relative vibration levels. The stable cuts are nearly silent, since the constant summed force of Sect. C.1 leaves this idealized slotting cut little to vibrate about; a real stable cut still carries runout and ambient sound, as Sect. C.2 showed. The chattering cuts are loud. The difference needs no instrumentation to detect.


In [ ]:
# --- Hearing the simulation: the four Example 4.11 cases on one loudness scale ---
def sonify(omega_rpm, b_mm, seconds=1.2, **kw):
    revs = int(np.ceil(seconds * omega_rpm / 60))
    s = mill_tds(omega_rpm=omega_rpm, b_mm=b_mm, revolutions=revs, **kw)
    a = resample_to_audio(s['x'] - s['x'].mean(), s['fs'])
    M, verdict = chatter_metric(s)
    return fade(a), verdict                 # unscaled; callers set loudness

clips = []
for lab in ('1', '2', '3', '4'):
    p4 = PRESETS[lab]
    a4, v4 = sonify(p4['omega_rpm'], p4['b_mm'])
    clips.append((lab, p4, a4, v4))
peak = max(np.max(np.abs(a4)) for _, _, a4, _ in clips)
for lab, p4, a4, v4 in clips:
    print(f"case {lab} ({p4['omega_rpm']:,} rpm, {p4['b_mm']} mm): "
          f"simulation verdict {v4}")
    display(Audio(0.8 * a4 / (peak + 1e-30), rate=AUDIO_RATE, normalize=False))


▸ **Key terms: time domain simulation, once-per-tooth sampling**


---

## C.6 Mapping Stability with the Simulation

The analytical boundaries of Sect. C.4 and the simulation of Sect. C.5 answer the same question by different routes, and the simulation can generate the entire map: run it over a grid of spindle speeds and depths, apply the periodic sampling metric to each run, and mark each grid point stable or chatter by the one micrometer line. The cell below does exactly that. The grid density and revolutions per point are parameters; the defaults trade resolution for a runtime of about a minute, with progress printed as it goes.


In [ ]:
# --- Build a stability map from time domain simulations. Adjust and rerun. ---
import time as _time
n_omega = 20          # grid points in spindle speed
n_b = 13              # grid points in axial depth
revs_map = 25         # revolutions per simulation (transient plus judgement)
print(f'Grid: {n_omega} speeds by {n_b} depths, {revs_map} revolutions each.')


In [ ]:
# --- Stability map sweep ---
om_grid = np.linspace(3000, 15000, n_omega)
b_grid = np.linspace(0.4, 6.5, n_b)

Mmap = np.zeros((n_b, n_omega))
t0 = _time.perf_counter()
for i_o, om in enumerate(om_grid):
    for i_b, bb in enumerate(b_grid):
        s = mill_tds(omega_rpm=om, b_mm=bb, revolutions=revs_map)
        M, verdict = chatter_metric(s)
        Mmap[i_b, i_o] = M
    print(f'  column {i_o + 1:2d}/{n_omega} ({om:,.0f} rpm) done, '
          f'{_time.perf_counter() - t0:5.1f} s elapsed')
print(f'{n_omega * n_b} simulations in {_time.perf_counter() - t0:.1f} s.')


In [ ]:
# --- Fig. C.7: the stability map ---
fig = go.Figure()
OMg, BBg = np.meshgrid(om_grid, b_grid)
stab = Mmap < 1.0
fig.add_trace(go.Scatter(x=OMg[stab], y=BBg[stab], mode='markers',
                         marker=dict(symbol='circle-open', size=8,
                                     color='#2ca02c', line=dict(width=1.5)),
                         name='stable (simulation)', customdata=Mmap[stab],
                         hovertemplate='%{x:.0f} rpm, %{y:.2f} mm<br>'
                                       'M = %{customdata:.2f} µm<extra></extra>'))
fig.add_trace(go.Scatter(x=OMg[~stab], y=BBg[~stab], mode='markers',
                         marker=dict(symbol='x', size=7, color='#d62728'),
                         name='chatter (simulation)', customdata=Mmap[~stab],
                         hovertemplate='%{x:.0f} rpm, %{y:.2f} mm<br>'
                                       'M = %{customdata:.2f} µm<extra></extra>'))
for i, (om, bl, fc) in enumerate(tl):
    fig.add_trace(go.Scatter(x=om, y=bl, mode='lines',
                             line=dict(color='#1f77b4'),
                             name='average tooth angle', showlegend=(i == 0)))
fig.add_trace(go.Scatter(x=zo_om, y=zo_bl, mode='lines',
                         line=dict(color='#ff7f0e', dash='dash'),
                         name='Fourier series (zero-order)'))
for label, om, b, verdict in CASES:
    fig.add_trace(go.Scatter(x=[om], y=[b], mode='markers+text',
                             marker=dict(color='black', size=9, symbol='diamond'),
                             text=[label], textposition='top center', showlegend=False))
fig.update_layout(title='Fig. C.7 — Stability map from the time domain simulation',
                  template='simple_white', height=500,
                  xaxis=dict(title='spindle speed (rpm)', range=[3000, 15000]),
                  yaxis=dict(title='b (mm)', range=[0, 6.5]),
                  legend=dict(x=0.02, y=0.98),
                  margin=dict(l=60, r=20, t=60, b=50))
fig.show()


The crosses fill the simulation's chatter region and the open circles its stable region. The zero-order boundary threads between them closely, pockets and low bands included, while the average tooth angle curve overstates the slotting pocket for the reason Sect. C.4 identified: with $\mu_y = 0$ it cannot see the y direction dynamics that the simulation retains. The comparison carries an honest engineering summary. The analytical solutions cost milliseconds and rest on averaging assumptions whose consequences vary with the cut geometry; the time domain map costs about a minute and keeps everything, including the nonlinearity of teeth leaving the cut. Where an analytical curve and the simulation disagree, the disagreement is diagnostic, and here it points directly at the discarded y direction.

<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
The once-per-tooth metric that judges each grid point is the same test applied to real cuts in practice: sample the vibration signal once per tooth passage and measure how much the samples move. A periodic, stable cut repeats itself every tooth period, so the samples hold still; chatter contains a frequency that is a multiple of no fundamental in the cut, so the samples wander. The samples are differenced and read in micrometers; the one micrometer dividing line used here reflects the contrast demonstrated in Sect. 4.4.5 of [1], near zero for stable cuts against tens of micrometers for chatter.
</div>

▸ **Key terms: stability map**


---

## C.7 Hearing Chatter

The detection rule assembles everything. A stable cut's spectrum contains only the families of Sect. C.2: the tooth passing frequency and its harmonics, the once-per-revolution runout family, and the idle baseline. Chatter adds a peak near a structural natural frequency that belongs to none of them [1]. The cell below analyzes the sonified case 2 cut from Sect. C.5 in the three views, with the tooth passing family marked; the chatter peak stands beside the family, close to it and not on it.


In [ ]:
# --- Fig. C.8: the chatter cut in three views, tooth passing family marked ---
audio_ch, _ = sonify(7500, 5.0)
audio_ch = 0.8 * audio_ch / (np.max(np.abs(audio_ch)) + 1e-30)
f_tooth = 500.0

f_sp, a_sp = compute_spectrum(audio_ch, AUDIO_RATE, fmax=2600)
t_g, f_g, S = compute_spectrogram(audio_ch, AUDIO_RATE, fmax=2600)
tt, xx = decimate_for_plot(np.arange(len(audio_ch)) / AUDIO_RATE, audio_ch,
                           max_points=6000)

fig = make_subplots(rows=1, cols=3,
                    column_titles=('time domain', 'spectrum', 'spectrogram'))
fig.add_trace(go.Scatter(x=tt, y=xx, mode='lines',
                         line=dict(color='#1f77b4', width=0.7),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=f_sp, y=a_sp, mode='lines',
                         line=dict(color='#1f77b4'), showlegend=False), row=1, col=2)
for m in range(1, 6):
    fig.add_shape(type='line', x0=m * f_tooth, x1=m * f_tooth, y0=0, y1=1.02,
                  xref='x2', yref='y2',
                  line=dict(color='black', width=1, dash='dot'), opacity=0.5)
fig.add_trace(go.Heatmap(x=t_g, y=f_g, z=S, colorscale='Inferno',
                         showscale=False), row=1, col=3)
fig.update_xaxes(title_text='t (s)', row=1, col=1)
fig.update_xaxes(title_text='frequency (Hz)', row=1, col=2)
fig.update_xaxes(title_text='t (s)', row=1, col=3)
fig.update_yaxes(title_text='amplitude', row=1, col=1)
fig.update_yaxes(title_text='amplitude (normalized)', range=[0, 1.05], row=1, col=2)
fig.update_yaxes(title_text='frequency (Hz)', range=[0, 2600], row=1, col=3)
fig.update_layout(title='Fig. C.8 — The chatter cut in three views',
                  template='simple_white', height=380,
                  margin=dict(l=60, r=20, t=70, b=50))
fig.show()

# identify the chatter frequency: the tallest peak away from the tooth family
mask = np.ones_like(f_sp, dtype=bool)
for m in range(1, 8):
    mask &= np.abs(f_sp - m * f_tooth) > 10
mask &= f_sp > 100
fc_detected = f_sp[mask][np.argmax(a_sp[mask])]
print(f'Dotted lines: the tooth passing family at multiples of {f_tooth:.0f} Hz.')
print(f'Tallest peak outside the family: {fc_detected:.0f} Hz, the chatter frequency, '
      f'near the {500:.0f} Hz natural frequency and on no family line.')


One caution before trusting the rule on a real machine. Runout populates the spectrum at multiples of the once-per-revolution frequency, and a runout harmonic can land close to a plausible chatter frequency; Example 6.4 of [1] shows a case where the third runout harmonic and the chatter frequency coincide and cannot be separated by frequency alone. The once-per-tooth sampling metric is the robust companion test, since runout content is periodic with the rotation and holds the samples steady, while chatter does not.

<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
This detection chain, spectrum, family masking, and once-per-tooth sampling, runs on live audio as well: a microphone supplies the signal, the programmed spindle speed and tooth count supply the families, and any persistent peak outside them is flagged with its frequency. That frequency is the input to the next section.
</div>

▸ **Key terms: chatter frequency**


---

## C.8 Choosing a Better Spindle Speed

The chatter frequency is a gift: it identifies where the structure wants to vibrate, and Eq. 4.24 turns it into a prescription. Setting $\varepsilon = 2\pi$, the best speeds place the tooth passing frequency in step with the chatter frequency:

$$\Omega_{new} = \frac{60 \, f_c}{(N + 1) \, N_t}$$

for lobe numbers $N = 0, 1, 2, \ldots$, which is the spindle speed regulation procedure of Sect. 6.1 of [1]. The new speed drives the tooth-to-tooth phase toward the favorable nesting of Fig. C.3, and because the chatter frequency shifts slightly at the new speed, the procedure may take an iteration or two to converge, exactly as Example 6.1 of [1] demonstrates. The cell below runs the whole loop on a cut that chatters: simulate, detect $f_c$ with the Sect. C.7 masking rule, compute the candidate speeds, pick the nearest, and repeat until the once-per-tooth metric reports a stable cut.


In [ ]:
# --- Speed selection: starting point. Adjust and rerun. ---
omega_0 = 6000.0     # starting spindle speed (rpm)
b_run = 2.5          # axial depth (mm), held fixed throughout
Nt_run = 4
fn_ref = 500.0       # the structure's natural frequency (Hz), for the search band
print(f'Starting point: {omega_0:,.0f} rpm at b = {b_run} mm.')


In [ ]:
# --- The listen, identify, adjust loop ---
def detect_fc(omega_rpm, b_mm):
    revs = int(np.ceil(0.9 * omega_rpm / 60))     # simulate about 0.9 s of cutting
    s = mill_tds(omega_rpm=omega_rpm, b_mm=b_mm, revolutions=revs)
    M, verdict = chatter_metric(s)
    half = len(s['x']) // 2
    f_sp, a_sp = compute_spectrum(s['x'][half:], s['fs'], fmax=2500)
    f_tp = omega_rpm * Nt_run / 60
    mask = f_sp > 0.95 * fn_ref                   # chatter sits near and above fn
    for m in range(1, 9):                         # remove the tooth passing family
        mask &= np.abs(f_sp - m * f_tp) > max(8.0, 0.015 * f_tp)
    fc = f_sp[mask][np.argmax(a_sp[mask])]
    return M, verdict, fc

omega = omega_0
history = []
for it in range(5):
    M, verdict, fc = detect_fc(omega, b_run)
    history.append((omega, verdict))
    print(f'iteration {it}: {omega:7,.0f} rpm -> metric M = {M:8.2f} µm ({verdict})'
          + ('' if verdict == 'stable' else f', chatter frequency {fc:6.1f} Hz'))
    if verdict == 'stable':
        break
    candidates = [60 * fc / ((N + 1) * Nt_run) for N in range(0, 4)]
    omega = min(candidates, key=lambda c: abs(np.log(c / omega)))
    print(f'             best speeds {", ".join(f"{c:,.0f}" for c in candidates)} rpm; '
          f'selecting {omega:,.0f} rpm')

print()
print(f'Converged: {history[-1][0]:,.0f} rpm at b = {b_run} mm, '
      f'from a starting point of {omega_0:,.0f} rpm.')
print('Listen to the first and last cuts of the loop:')
a0, _ = sonify(history[0][0], b_run)
a1, _ = sonify(history[-1][0], b_run)
peak = max(np.max(np.abs(a0)), np.max(np.abs(a1)))
a0 = 0.8 * a0 / (peak + 1e-30)
a1 = 0.8 * a1 / (peak + 1e-30)
print(f'before, {history[0][0]:,.0f} rpm:')
display(Audio(a0, rate=AUDIO_RATE, normalize=False))
print(f'after, {history[-1][0]:,.0f} rpm:')
display(Audio(a1, rate=AUDIO_RATE, normalize=False))


The loop lands just above 7,500 rpm, beside the stable pocket that case 1 of Example 4.11 occupies, where the tooth passing frequency falls in step with the structure's preferred frequency. With a deeper cut the first adjusted speed can itself chatter at a shifted chatter frequency, and the loop simply repeats; Example 6.1 of [1] shows exactly this behavior before converging. The audio pair summarizes the change: self-excited vibration at the first speed and forced vibration at the second, with nothing changed but the spindle speed. On the map of Fig. C.7, the operating point has moved horizontally from the chatter region into a pocket, and Fig. C.4 explains why the pocket is there.

▸ **Key terms: best speeds, spindle speed regulation**


---

## Summary

- Milling removes material with a rotating, toothed tool; the chip thickness varies as $h = f_t \sin\phi$ (Eq. 4.2) over the engagement window set by the radial immersion, and the cutting force follows it through the coefficients $k_t$ and $k_n$ (Eqs. 4.7 to 4.10).
- The cutting force is periodic with fundamental frequency $f_{tooth} = \Omega N_t / 60$ (Eq. 4.14), so a stable cut's spectrum contains the tooth passing family, the once-per-revolution runout family, and the idle baseline, and every healthy line has a name.
- Regeneration closes a feedback loop: each tooth cuts the surface the previous tooth left (Eqs. 4.21 and 4.22), and the tooth-to-tooth phase decides whether vibration is harmlessly copied or pumped into chatter.
- The stability lobe diagram gives the limiting depth $b_{lim}(\Omega)$; two analytical solutions compute it, the average tooth angle approach (Eqs. 4.23, 4.24, 4.27) and the Fourier series zero-order approach (Eqs. 4.44, 4.45). For slotting they disagree, because $\mu_y = 0$ removes the y direction from the average tooth angle result, and the zero-order boundary is the one the simulation confirms.
- The time domain simulation of Sect. 4.4 keeps the full geometry and the out-of-cut nonlinearity; its four Example 4.11 cases reproduce the Example 4.11 verdicts of [1], and sweeping it over a grid produces a stability map that the analytical boundaries trace.
- Once-per-tooth sampling separates stable cuts from chatter: periodic cuts hold the samples steady, chatter scatters them, and the test is robust to runout content that frequency masking alone cannot separate.
- The chatter frequency prescribes better spindle speeds through $\Omega_{new} = 60 f_c / ((N+1) N_t)$, and iterating the listen, identify, adjust loop drives the cut into a stable pocket.

The series ends where it aimed: the stability lobe diagram computed by two analytical routes, checked by a time domain simulation, and navigated by listening. The same procedure applies at a machine within earshot: record the cut, remove the tooth passing family from the spectrum, take a persistent remaining peak as the chatter frequency, and move the spindle speed to place a tooth passing harmonic on it.


---

## Exercises

**1.** Using the parameter panel of Sect. C.5, change the feed per tooth from 0.15 to 0.05 mm/tooth for case 2.
**(a)** Does the cut remain in chatter?
**(b)** Explain, using Eqs. 4.7 and 4.22, why the feed per tooth scales the forces and vibrations but does not decide stability.

**2.** Run case 2 with `steps_rev` set to 200, 652, and 2000.
**(a)** How do the verdict and the displacement histories change?
**(b)** What does this suggest as a working rule for choosing the number of steps per revolution?

**3.** In Fig. C.5, the whole diagram scales with the damping ratio.
**(a)** Using Eq. 4.23 and the Part A result that the FRF peak scales as $1/(2 k \zeta)$, show that the critical (minimum) limiting depth is proportional to $\zeta$ for small $\zeta$.
**(b)** A tooling change doubles the damping ratio. What happens to the deepest stable slotting cut at the worst spindle speed?

**4.** Compute both analytical boundaries for a 25 percent radial immersion up milling cut ($\phi_s = 0$, $\phi_e = 60$ deg) using the Sect. C.4 cells.
**(a)** How do the two solutions compare with each other at this low immersion, and with the slotting result?
**(b)** Chap. 4 of [1] cautions that averaging degrades as the force becomes impulse-like. Which figure of Sect. C.1 shows why?

**5.** Rebuild the Sect. C.6 map with `zeta_x = zeta_y = 0.04` (edit `mill_tds`'s defaults or pass the values through).
**(a)** Compare the chatter region with Fig. C.7.
**(b)** Do the analytical boundaries recomputed at the new damping still trace its edge?

**6.** Retune the Sect. C.8 loop for a different structure, the system of Example 6.1 of [1] made symmetric: in the visible starting point cell set `omega_0 = 10000`, `b_run = 1.5`, and `fn_ref = 800`, and in the collapsed loop cell add `fn_x=800, fn_y=800, k_x=5e6, k_y=5e6, zeta_x=0.01, zeta_y=0.01` to the `mill_tds` call inside `detect_fc`.
**(a)** Before running, compute the best speeds $60 f_n / ((N+1) N_t)$ for $N = 0$ and $N = 1$.
**(b)** Run the loop. Where does it land relative to your predictions, and how many iterations does it take?


---

## Appendix: Utility Functions

The shared helper functions, defined once in the collapsed cell at the top of the notebook, are carried over from Parts A and B: `fade`, `compute_spectrum`, `compute_spectrogram`, and `decimate_for_plot`, plus one addition, `resample_to_audio(x, fs_in, fs_out)`, which interpolates a simulated record onto the audio sample grid for playback. The stability solutions `tlusty_lobes` and `zoa_lobes` and the simulation `mill_tds` with its companion `chatter_metric` are defined in full view in Sects. C.4 and C.5, because they are the subject matter of this Part rather than plumbing.


---

## References

**[1]** Schmitz, T. L., and Smith, K. S. (2019). *Machining Dynamics: Frequency Response to Improved Productivity*, 2nd ed. Springer. DOI: 10.1007/978-3-319-93707-6

**[2]** Altintas, Y., and Budak, E. (1995). Analytical prediction of stability lobes in milling. *CIRP Annals*, 44(1), 357 to 362.

**[3]** Taylor, F. W. (1907). On the art of cutting metals. *Transactions of the ASME*, 28, 31 to 350.

**[4]** Schmitz, T. L., and Gomez, M. F. *Machining Dynamics: Jupyter Notebook Edition* (in preparation).
